# Molecule Properties Prediction

## Machine Learning

In [12]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (recall_score, accuracy_score, confusion_matrix)
from sklearn.base import clone
from lightgbm import LGBMClassifier
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from tqdm import tqdm

In [13]:
## load in dataset
tox_data = pd.read_csv('../data/tox21.csv')
tox_data.head(2)

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O


In [14]:
nbt = 2048
threshold = 0.5

#### 1. Data Processing

In [15]:
# Step 1: Identify label columns (12 toxicity-related labels)
label_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]

# Step 2: Count the number of samples with completely missing or non-missing labels
total_samples = tox_data.shape[0]
all_nan_labels = tox_data[label_cols].isna().all(axis=1).sum()
no_nan_labels = tox_data[label_cols].notna().all(axis=1).sum()
partial_nan_labels = total_samples - all_nan_labels - no_nan_labels

{
    "Total number of samples": total_samples,
    "Samples with all labels missing": int(all_nan_labels),
    "Samples with no missing labels": int(no_nan_labels),
    "Samples with partially missing labels": int(partial_nan_labels)
}

# Count the number of missing values for each label
missing_counts_per_label = tox_data[label_cols].isna().sum().sort_values(ascending=False)

missing_counts_per_label

# ## Option: Fill missing label values with 1 (assume absence of toxicity feature)
# # Fill all missing values in label columns with 1, indicating "non-toxic"
# tox_data[label_cols] = tox_data[label_cols].fillna(1)

# # Check whether the filling was successful (should return 0)
# remaining_missing = tox_data[label_cols].isna().sum().sum()
# int(remaining_missing)

SR-MMP           2092
SR-ARE           2078
NR-Aromatase     2072
NR-ER            1697
NR-PPAR-gamma    1430
SR-HSE           1419
NR-AhR           1322
NR-AR-LBD        1111
SR-p53           1104
NR-ER-LBD         901
SR-ATAD5          781
NR-AR             574
dtype: int64

#### 2. Feature Engineering

In [16]:
# # Define a function to convert SMILES to ECFP4 fingerprint vector
# def smiles_to_ecfp4(smiles, radius=2, nBits=nbt):
#     mol = Chem.MolFromSmiles(smiles)  # Use RDKit to convert SMILES into a molecule object (Mol)
#     if mol is None:
#         return np.zeros(nBits, dtype=int)  # Return an all-zero vector if SMILES is invalid

#     fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)  
#     # Generate ECFP4 fingerprint; radius=2 means ECFP4, nBits specifies fingerprint length (e.g., 1024 bits)

#     arr = np.zeros((nBits,), dtype=int)  # Create a one-dimensional array of all zeros
#     DataStructs.ConvertToNumpyArray(fp, arr)  # Copy the fingerprint BitVect to the NumPy array
#     return arr  # Return a NumPy binary vector (0/1) with length nBits



def smiles_to_ecfp4(smiles, radius=2, nBits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nBits, dtype=int)
    
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits)
    fp = gen.GetFingerprint(mol)

    arr = np.zeros((nBits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


# Apply the fingerprint extraction with progress bar
tqdm.pandas()
X_fp = np.vstack(tox_data['smiles'].progress_apply(smiles_to_ecfp4))

# Label matrix
y = tox_data[label_cols].values

# Output the shapes of the resulting matrices
print("Feature matrix X_fp.shape =", X_fp.shape)
print("Label matrix y.shape =", y.shape)

100%|██████████| 8006/8006 [00:02<00:00, 3716.38it/s]

Feature matrix X_fp.shape = (8006, 1024)
Label matrix y.shape = (8006, 12)


#### 3. Models

Binary Relevance Strategy

In [18]:
def recall_with_custom_threshold(threshold):
    def scorer(estimator, X, y):
        y_proba = estimator.predict_proba(X)[:, 1]
        y_pred = (y_proba > threshold).astype(int)
        return recall_score(y, y_pred)
    return scorer


def auc_with_proba():
    def scorer(estimator, X, y):
        y_proba = estimator.predict_proba(X)[:, 1]
        return roc_auc_score(y, y_proba)
    return scorer


def custom_auc(y_true, y_proba):
    if len(np.unique(y_true)) == 1:
        return 0.5  # fallback for constant labels
    return roc_auc_score(y_true, y_proba)

auc_scorer = make_scorer(custom_auc, needs_proba=True)

1. XGBoost

##### a.threshold = 0.5
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Positive Rate</th>
      <th>TP Rate</th>
      <th>FP Rate</th>
      <th>Recall</th>
      <th>Precision</th>
      <th>F1 Score</th>
      <th>Accuracy</th>
      <th>AUC</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>NR-AhR</th>
      <td>0.1174</td>
      <td>0.7624</td>
      <td>0.2376</td>
      <td>0.4904</td>
      <td>0.7624</td>
      <td>0.5969</td>
      <td>0.9222</td>
      <td>0.9262</td>
    </tr>
    <tr>
      <th>SR-MMP</th>
      <td>0.1581</td>
      <td>0.7045</td>
      <td>0.2955</td>
      <td>0.4973</td>
      <td>0.7045</td>
      <td>0.5831</td>
      <td>0.8876</td>
      <td>0.8832</td>
    </tr>
    <tr>
      <th>NR-AR-LBD</th>
      <td>0.0348</td>
      <td>0.7812</td>
      <td>0.2188</td>
      <td>0.5208</td>
      <td>0.7812</td>
      <td>0.6250</td>
      <td>0.9782</td>
      <td>0.8491</td>
    </tr>
    <tr>
      <th>SR-ATAD5</th>
      <td>0.0367</td>
      <td>0.5926</td>
      <td>0.4074</td>
      <td>0.3019</td>
      <td>0.5926</td>
      <td>0.4000</td>
      <td>0.9668</td>
      <td>0.8335</td>
    </tr>
    <tr>
      <th>NR-ER-LBD</th>
      <td>0.0500</td>
      <td>0.7561</td>
      <td>0.2439</td>
      <td>0.4366</td>
      <td>0.7561</td>
      <td>0.5536</td>
      <td>0.9648</td>
      <td>0.8171</td>
    </tr>
    <tr>
      <th>SR-p53</th>
      <td>0.0623</td>
      <td>0.6471</td>
      <td>0.3529</td>
      <td>0.2558</td>
      <td>0.6471</td>
      <td>0.3667</td>
      <td>0.9450</td>
      <td>0.8089</td>
    </tr>
    <tr>
      <th>SR-HSE</th>
      <td>0.0577</td>
      <td>0.5000</td>
      <td>0.5000</td>
      <td>0.1316</td>
      <td>0.5000</td>
      <td>0.2083</td>
      <td>0.9423</td>
      <td>0.8081</td>
    </tr>
    <tr>
      <th>SR-ARE</th>
      <td>0.1619</td>
      <td>0.5772</td>
      <td>0.4228</td>
      <td>0.3698</td>
      <td>0.5772</td>
      <td>0.4508</td>
      <td>0.8541</td>
      <td>0.7877</td>
    </tr>
    <tr>
      <th>NR-AR</th>
      <td>0.0417</td>
      <td>0.5814</td>
      <td>0.4186</td>
      <td>0.4032</td>
      <td>0.5814</td>
      <td>0.4762</td>
      <td>0.9630</td>
      <td>0.7683</td>
    </tr>
    <tr>
      <th>NR-PPAR-gamma</th>
      <td>0.0289</td>
      <td>0.5385</td>
      <td>0.4615</td>
      <td>0.1842</td>
      <td>0.5385</td>
      <td>0.2745</td>
      <td>0.9719</td>
      <td>0.7328</td>
    </tr>
    <tr>
      <th>NR-Aromatase</th>
      <td>0.0514</td>
      <td>0.5185</td>
      <td>0.4815</td>
      <td>0.2295</td>
      <td>0.5185</td>
      <td>0.3182</td>
      <td>0.9495</td>
      <td>0.7286</td>
    </tr>
    <tr>
      <th>NR-ER</th>
      <td>0.1260</td>
      <td>0.5469</td>
      <td>0.4531</td>
      <td>0.2201</td>
      <td>0.5469</td>
      <td>0.3139</td>
      <td>0.8788</td>
      <td>0.6720</td>
    </tr>
    <tr>
      <th>Overall (mean)</th>
      <td>0.0772</td>
      <td>0.6255</td>
      <td>0.3745</td>
      <td>0.3368</td>
      <td>0.6255</td>
      <td>0.4306</td>
      <td>0.9354</td>
      <td>0.8013</td>
    </tr>
    <tr>
      <th>Overall (median)</th>
      <td>0.0546</td>
      <td>0.5870</td>
      <td>0.4130</td>
      <td>0.3358</td>
      <td>0.5870</td>
      <td>0.4254</td>
      <td>0.9472</td>
      <td>0.8085</td>
    </tr>
  </tbody>
</table>
</div>

##### b.threshold
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Positive Rate</th>
      <th>TP Rate</th>
      <th>FP Rate</th>
      <th>Recall</th>
      <th>Precision</th>
      <th>F1 Score</th>
      <th>Accuracy</th>
      <th>AUC</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>NR-AhR</th>
      <td>0.1174</td>
      <td>0.6528</td>
      <td>0.3472</td>
      <td>0.5987</td>
      <td>0.6528</td>
      <td>0.6246</td>
      <td>0.9155</td>
      <td>0.9210</td>
    </tr>
    <tr>
      <th>SR-MMP</th>
      <td>0.1581</td>
      <td>0.6685</td>
      <td>0.3315</td>
      <td>0.6578</td>
      <td>0.6685</td>
      <td>0.6631</td>
      <td>0.8943</td>
      <td>0.8952</td>
    </tr>
    <tr>
      <th>SR-ATAD5</th>
      <td>0.0367</td>
      <td>0.7407</td>
      <td>0.2593</td>
      <td>0.3774</td>
      <td>0.7407</td>
      <td>0.5000</td>
      <td>0.9723</td>
      <td>0.8446</td>
    </tr>
    <tr>
      <th>NR-AR-LBD</th>
      <td>0.0348</td>
      <td>0.7353</td>
      <td>0.2647</td>
      <td>0.5208</td>
      <td>0.7353</td>
      <td>0.6098</td>
      <td>0.9768</td>
      <td>0.8031</td>
    </tr>
    <tr>
      <th>SR-HSE</th>
      <td>0.0577</td>
      <td>0.3333</td>
      <td>0.6667</td>
      <td>0.0921</td>
      <td>0.3333</td>
      <td>0.1443</td>
      <td>0.9370</td>
      <td>0.7821</td>
    </tr>
    <tr>
      <th>SR-ARE</th>
      <td>0.1619</td>
      <td>0.4785</td>
      <td>0.5215</td>
      <td>0.4635</td>
      <td>0.4785</td>
      <td>0.4709</td>
      <td>0.8314</td>
      <td>0.7760</td>
    </tr>
    <tr>
      <th>SR-p53</th>
      <td>0.0623</td>
      <td>0.5306</td>
      <td>0.4694</td>
      <td>0.3023</td>
      <td>0.5306</td>
      <td>0.3852</td>
      <td>0.9399</td>
      <td>0.7757</td>
    </tr>
    <tr>
      <th>NR-ER-LBD</th>
      <td>0.0500</td>
      <td>0.6957</td>
      <td>0.3043</td>
      <td>0.4507</td>
      <td>0.6957</td>
      <td>0.5470</td>
      <td>0.9627</td>
      <td>0.7753</td>
    </tr>
    <tr>
      <th>NR-Aromatase</th>
      <td>0.0514</td>
      <td>0.5000</td>
      <td>0.5000</td>
      <td>0.2295</td>
      <td>0.5000</td>
      <td>0.3146</td>
      <td>0.9486</td>
      <td>0.7716</td>
    </tr>
    <tr>
      <th>NR-AR</th>
      <td>0.0417</td>
      <td>0.6579</td>
      <td>0.3421</td>
      <td>0.4032</td>
      <td>0.6579</td>
      <td>0.5000</td>
      <td>0.9664</td>
      <td>0.7417</td>
    </tr>
    <tr>
      <th>NR-PPAR-gamma</th>
      <td>0.0289</td>
      <td>0.6667</td>
      <td>0.3333</td>
      <td>0.2632</td>
      <td>0.6667</td>
      <td>0.3774</td>
      <td>0.9749</td>
      <td>0.7182</td>
    </tr>
    <tr>
      <th>NR-ER</th>
      <td>0.1260</td>
      <td>0.4653</td>
      <td>0.5347</td>
      <td>0.2956</td>
      <td>0.4653</td>
      <td>0.3615</td>
      <td>0.8685</td>
      <td>0.6743</td>
    </tr>
    <tr>
      <th>Overall (mean)</th>
      <td>0.0772</td>
      <td>0.5938</td>
      <td>0.4062</td>
      <td>0.3879</td>
      <td>0.5938</td>
      <td>0.4582</td>
      <td>0.9324</td>
      <td>0.7899</td>
    </tr>
    <tr>
      <th>Overall (median)</th>
      <td>0.0546</td>
      <td>0.6554</td>
      <td>0.3446</td>
      <td>0.3903</td>
      <td>0.6554</td>
      <td>0.4854</td>
      <td>0.9442</td>
      <td>0.7758</td>
    </tr>
  </tbody>
</table>
</div>


Grid Search <br>
- Applied to the label found to be the least accurate: `"SR-ATAD5"`

In [20]:
# Set target label
target_label = "SR-ATAD5"

# Filter samples with non-missing values for the target label
valid_idx = ~tox_data[target_label].isna()
X_valid = X_fp[valid_idx]
y_valid = tox_data[target_label].values[valid_idx]

print(f"Starting hyperparameter tuning for {target_label}, valid samples: {len(y_valid)}")

# Split into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(
    X_valid, y_valid, test_size=0.2, random_state=5104, stratify=y_valid)

Starting hyperparameter tuning for SR-ATAD5, valid samples: 7225


In [21]:
param_grid = {
    'max_depth': [5, 10 , 15],
    'n_estimators': [500 , 700 , 900],
    'learning_rate': [0.05, 0.1 ,0.12]
}



# GridSearchCV 
grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss'),
    param_grid,
    # scoring= recall_with_custom_threshold(threshold),
    scoring= auc_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)

# GridSearch
grid.fit(X_train, y_train)

# best parameters
print("\n best parameter:")
print(grid.best_params_)

y_pred = grid.best_estimator_.predict(X_test)

# print the auc on the testset

print(f"\n Recall on testset: {recall_score(y_test, y_pred):.4f}")

Fitting 3 folds for each of 27 candidates, totalling 81 fits

 best parameter:
{'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 500}

 Recall on testset: 0.2075


Fit and Predict
- Fit the model using the best parameters and make predictions.

In [22]:
X = X_fp
y = tox_data[label_cols].values
label_names = label_cols

In [23]:
from sklearn.metrics import (
    confusion_matrix, recall_score, accuracy_score,
    precision_score, roc_auc_score, f1_score
)

# Initialize containers
models = {}
metrics = {}

# Collect predictions and truths for overall evaluation
all_y_true = []
all_y_pred = []
all_y_proba = []

for label in label_names:
    valid_idx = ~tox_data[label].isna()
    X_valid = X_fp[valid_idx]
    y_valid = tox_data[label].values[valid_idx]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid,
        test_size=0.2,
        random_state=5104,
        stratify=y_valid
    )

    # model = clone(grid.best_estimator_)
    # model.set_params(verbosity=0)  # Silence XGBoost output
    # model.fit(X_train, y_train)
    
    # best parameter got from grid search
    best_parameter = {
    'n_estimators': 500,
    'max_depth': 5,
    'learning_rate': 0.05,
    'random_state': 5104,
    'verbosity': 0}

    model = XGBClassifier(**best_parameter)
    model.fit(X_train, y_train)
    
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    positive_rate = np.mean(y_test)
    tp_rate = tp / (tp + fp) if (tp + fp) > 0 else 0
    fp_rate = fp / (tp + fp) if (tp + fp) > 0 else 0
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    models[label] = model
    metrics[label] = {
        "Positive Rate": round(positive_rate, 4),
        "TP Rate": round(tp_rate, 4),
        "FP Rate": round(fp_rate, 4),
        "Recall": round(recall, 4),
        "Precision": round(precision, 4),
        "F1 Score": round(f1, 4),
        "Accuracy": round(accuracy, 4),
        "AUC": round(auc, 4)
    }

    # Collect for overall metrics
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_proba.extend(y_proba)

# Convert metrics to DataFrame and sort
model_XGBoost = pd.DataFrame(metrics).T.sort_values(by="AUC", ascending=False)

# Calculate mean and median of all metrics across labels
overall_mean = model_XGBoost.astype(float).mean().round(4).to_dict()
overall_median = model_XGBoost.astype(float).median().round(4).to_dict()

# Add two summary rows: mean and median across all labels
model_XGBoost.loc["Overall (mean)"] = overall_mean
model_XGBoost.loc["Overall (median)"] = overall_median

# Display the final evaluation table
display(model_XGBoost)

,Positive Rate,TP Rate,FP Rate,Recall,Precision,F1 Score,Accuracy,AUC
NR-AhR,0.1174,0.7500,0.2500,0.4586,0.7500,0.5692,0.9185,0.9174
SR-MMP,0.1581,0.7477,0.2523,0.4439,0.7477,0.5570,0.8884,0.8930
SR-ATAD5,0.0367,0.8462,0.1538,0.2075,0.8462,0.3333,0.9696,0.8371
NR-AR-LBD,0.0348,0.8462,0.1538,0.4583,0.8462,0.5946,0.9782,0.8315
NR-ER-LBD,0.0500,0.8148,0.1852,0.3099,0.8148,0.4490,0.9620,0.7946
SR-p53,0.0623,0.8421,0.1579,0.1860,0.8421,0.3048,0.9471,0.7907
SR-ARE,0.1619,0.6494,0.3506,0.2604,0.6494,0.3717,0.8575,0.7806
NR-Aromatase,0.0514,0.7333,0.2667,0.1803,0.7333,0.2895,0.9545,0.7758
SR-HSE,0.0577,0.8000,0.2000,0.0526,0.8000,0.0988,0.9446,0.7717
NR-AR,0.0417,0.8929,0.1071,0.4032,0.8929,0.5556,0.9731,0.7642


##### 2. LightGBM

##### a.threshold = 0.5
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Positive Rate</th>
      <th>TP Rate</th>
      <th>FP Rate</th>
      <th>Recall</th>
      <th>Precision</th>
      <th>F1 Score</th>
      <th>Accuracy</th>
      <th>AUC</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>NR-AhR</th>
      <td>0.1174</td>
      <td>0.7308</td>
      <td>0.2692</td>
      <td>0.3631</td>
      <td>0.7308</td>
      <td>0.4851</td>
      <td>0.9095</td>
      <td>0.8821</td>
    </tr>
    <tr>
      <th>SR-MMP</th>
      <td>0.1581</td>
      <td>0.7353</td>
      <td>0.2647</td>
      <td>0.4011</td>
      <td>0.7353</td>
      <td>0.5190</td>
      <td>0.8825</td>
      <td>0.8406</td>
    </tr>
    <tr>
      <th>SR-p53</th>
      <td>0.0623</td>
      <td>0.8000</td>
      <td>0.2000</td>
      <td>0.1395</td>
      <td>0.8000</td>
      <td>0.2376</td>
      <td>0.9442</td>
      <td>0.8375</td>
    </tr>
    <tr>
      <th>SR-ATAD5</th>
      <td>0.0367</td>
      <td>0.7500</td>
      <td>0.2500</td>
      <td>0.1698</td>
      <td>0.7500</td>
      <td>0.2769</td>
      <td>0.9675</td>
      <td>0.8313</td>
    </tr>
    <tr>
      <th>NR-AR-LBD</th>
      <td>0.0348</td>
      <td>0.8065</td>
      <td>0.1935</td>
      <td>0.5208</td>
      <td>0.8065</td>
      <td>0.6329</td>
      <td>0.9790</td>
      <td>0.8197</td>
    </tr>
    <tr>
      <th>NR-Aromatase</th>
      <td>0.0514</td>
      <td>0.7619</td>
      <td>0.2381</td>
      <td>0.2623</td>
      <td>0.7619</td>
      <td>0.3902</td>
      <td>0.9579</td>
      <td>0.7969</td>
    </tr>
    <tr>
      <th>SR-ARE</th>
      <td>0.1619</td>
      <td>0.7667</td>
      <td>0.2333</td>
      <td>0.2396</td>
      <td>0.7667</td>
      <td>0.3651</td>
      <td>0.8651</td>
      <td>0.7885</td>
    </tr>
    <tr>
      <th>NR-ER-LBD</th>
      <td>0.0500</td>
      <td>0.7586</td>
      <td>0.2414</td>
      <td>0.3099</td>
      <td>0.7586</td>
      <td>0.4400</td>
      <td>0.9606</td>
      <td>0.7827</td>
    </tr>
    <tr>
      <th>NR-PPAR-gamma</th>
      <td>0.0289</td>
      <td>0.7500</td>
      <td>0.2500</td>
      <td>0.1579</td>
      <td>0.7500</td>
      <td>0.2609</td>
      <td>0.9742</td>
      <td>0.7793</td>
    </tr>
    <tr>
      <th>SR-HSE</th>
      <td>0.0577</td>
      <td>0.7083</td>
      <td>0.2917</td>
      <td>0.2237</td>
      <td>0.7083</td>
      <td>0.3400</td>
      <td>0.9499</td>
      <td>0.7376</td>
    </tr>
    <tr>
      <th>NR-AR</th>
      <td>0.0417</td>
      <td>0.8462</td>
      <td>0.1538</td>
      <td>0.3548</td>
      <td>0.8462</td>
      <td>0.5000</td>
      <td>0.9704</td>
      <td>0.7118</td>
    </tr>
    <tr>
      <th>NR-ER</th>
      <td>0.1260</td>
      <td>0.7083</td>
      <td>0.2917</td>
      <td>0.2138</td>
      <td>0.7083</td>
      <td>0.3285</td>
      <td>0.8899</td>
      <td>0.7006</td>
    </tr>
    <tr>
      <th>Overall (mean)</th>
      <td>0.0772</td>
      <td>0.7602</td>
      <td>0.2398</td>
      <td>0.2797</td>
      <td>0.7602</td>
      <td>0.3980</td>
      <td>0.9376</td>
      <td>0.7924</td>
    </tr>
    <tr>
      <th>Overall (median)</th>
      <td>0.0546</td>
      <td>0.7543</td>
      <td>0.2457</td>
      <td>0.2510</td>
      <td>0.7543</td>
      <td>0.3776</td>
      <td>0.9539</td>
      <td>0.7927</td>
    </tr>
  </tbody>
</table>
</div>


##### b.threshold = 0.3
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Positive Rate</th>
      <th>TP Rate</th>
      <th>FP Rate</th>
      <th>Recall</th>
      <th>Precision</th>
      <th>F1 Score</th>
      <th>Accuracy</th>
      <th>AUC</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>NR-AhR</th>
      <td>0.1174</td>
      <td>0.5915</td>
      <td>0.4085</td>
      <td>0.5350</td>
      <td>0.5915</td>
      <td>0.5619</td>
      <td>0.9020</td>
      <td>0.8811</td>
    </tr>
    <tr>
      <th>SR-MMP</th>
      <td>0.1581</td>
      <td>0.5778</td>
      <td>0.4222</td>
      <td>0.5561</td>
      <td>0.5778</td>
      <td>0.5668</td>
      <td>0.8656</td>
      <td>0.8540</td>
    </tr>
    <tr>
      <th>NR-AR-LBD</th>
      <td>0.0348</td>
      <td>0.7073</td>
      <td>0.2927</td>
      <td>0.6042</td>
      <td>0.7073</td>
      <td>0.6517</td>
      <td>0.9775</td>
      <td>0.8325</td>
    </tr>
    <tr>
      <th>SR-p53</th>
      <td>0.0623</td>
      <td>0.6154</td>
      <td>0.3846</td>
      <td>0.2791</td>
      <td>0.6154</td>
      <td>0.3840</td>
      <td>0.9442</td>
      <td>0.8325</td>
    </tr>
    <tr>
      <th>SR-ATAD5</th>
      <td>0.0367</td>
      <td>0.7222</td>
      <td>0.2778</td>
      <td>0.2453</td>
      <td>0.7222</td>
      <td>0.3662</td>
      <td>0.9689</td>
      <td>0.8095</td>
    </tr>
    <tr>
      <th>SR-ARE</th>
      <td>0.1619</td>
      <td>0.5500</td>
      <td>0.4500</td>
      <td>0.4583</td>
      <td>0.5500</td>
      <td>0.5000</td>
      <td>0.8516</td>
      <td>0.7907</td>
    </tr>
    <tr>
      <th>NR-Aromatase</th>
      <td>0.0514</td>
      <td>0.7308</td>
      <td>0.2692</td>
      <td>0.3115</td>
      <td>0.7308</td>
      <td>0.4368</td>
      <td>0.9587</td>
      <td>0.7881</td>
    </tr>
    <tr>
      <th>NR-ER-LBD</th>
      <td>0.0500</td>
      <td>0.6190</td>
      <td>0.3810</td>
      <td>0.3662</td>
      <td>0.6190</td>
      <td>0.4602</td>
      <td>0.9571</td>
      <td>0.7690</td>
    </tr>
    <tr>
      <th>NR-PPAR-gamma</th>
      <td>0.0289</td>
      <td>0.5833</td>
      <td>0.4167</td>
      <td>0.1842</td>
      <td>0.5833</td>
      <td>0.2800</td>
      <td>0.9726</td>
      <td>0.7441</td>
    </tr>
    <tr>
      <th>SR-HSE</th>
      <td>0.0577</td>
      <td>0.7000</td>
      <td>0.3000</td>
      <td>0.2763</td>
      <td>0.7000</td>
      <td>0.3962</td>
      <td>0.9514</td>
      <td>0.7316</td>
    </tr>
    <tr>
      <th>NR-ER</th>
      <td>0.1260</td>
      <td>0.4370</td>
      <td>0.5630</td>
      <td>0.3270</td>
      <td>0.4370</td>
      <td>0.3741</td>
      <td>0.8621</td>
      <td>0.6902</td>
    </tr>
    <tr>
      <th>NR-AR</th>
      <td>0.0417</td>
      <td>0.7742</td>
      <td>0.2258</td>
      <td>0.3871</td>
      <td>0.7742</td>
      <td>0.5161</td>
      <td>0.9697</td>
      <td>0.6892</td>
    </tr>
    <tr>
      <th>Overall (mean)</th>
      <td>0.0772</td>
      <td>0.6340</td>
      <td>0.3660</td>
      <td>0.3775</td>
      <td>0.6340</td>
      <td>0.4578</td>
      <td>0.9318</td>
      <td>0.7844</td>
    </tr>
    <tr>
      <th>Overall (median)</th>
      <td>0.0546</td>
      <td>0.6172</td>
      <td>0.3828</td>
      <td>0.3466</td>
      <td>0.6172</td>
      <td>0.4485</td>
      <td>0.9542</td>
      <td>0.7894</td>
    </tr>
  </tbody>
</table>
</div>

Grid Search  
- Applied to the label found to be the least accurate: `"SR-ATAD5"`

In [24]:
# Set target label
target_label = "SR-ATAD5"

# Filter samples with non-missing values for the target label
valid_idx = ~tox_data[target_label].isna()
X_valid = X_fp[valid_idx]
y_valid = tox_data[target_label].values[valid_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_valid, y_valid, test_size=0.2, random_state=5104, stratify=y_valid
)

In [25]:
# Grid search with LightGBM
grid = GridSearchCV(
    LGBMClassifier(eval_metric='logloss', verbosity=-1),  # Fully suppress logs and warnings
    param_grid={
        'n_estimators': [300, 500, 700],
        'max_depth': [8, 10, 13, 15],
        'learning_rate': [0.05, 0.1]
    },
    # scoring=recall_with_custom_threshold(threshold),
    scoring=auc_with_proba(),  # Make sure this function is silent
    cv=3,
    verbose=0,  # Suppress GridSearchCV logs too
    n_jobs=-1
)

# Fit the grid search
grid.fit(X_train, y_train)

# Print best parameters
print(" Best hyperparameters:", grid.best_params_)

 Best hyperparameters: {'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 300}


Fit and Predict
- Fit the model using the best parameters and make predictions.

In [26]:
models = {}
metrics = {}

# Store all predictions and ground truths across labels (for overall evaluation)
all_y_true = []
all_y_pred = []
all_y_proba = []

for label in label_names:
    # Select samples with non-missing values for the current label
    valid_idx = ~tox_data[label].isna()
    X_valid = X_fp[valid_idx]
    y_valid = tox_data[label].values[valid_idx]

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid, test_size=0.2, random_state=42, stratify=y_valid
    )

    # # model = LGBMClassifier(**best_params)
    # model = clone(grid.best_estimator_)
    # model.set_params(verbosity=-1)  # Optional: suppress LightGBM warnings
    # model.fit(X_train, y_train)
    
    # best_params got from grid search
    best_params = {
    'n_estimators': 300,
    'max_depth': 8,
    'learning_rate': 0.05,
     'random_state': 5104,
    'verbosity': -1 # Optional: suppress LightGBM warnings
    }

    model = LGBMClassifier(**best_params)
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    positive_rate = np.mean(y_test)
    tp_rate = tp / (tp + fp) if (tp + fp) > 0 else 0
    fp_rate = fp / (tp + fp) if (tp + fp) > 0 else 0
    
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    # Store the trained model and evaluation metrics
    models[label] = model
    metrics[label] = {
        "Positive Rate": round(positive_rate, 4),
        "TP Rate": round(tp_rate, 4),
        "FP Rate": round(fp_rate, 4),
        "Recall": round(recall, 4),
        "Precision": round(precision, 4),
        "F1 Score": round(f1, 4),
        "Accuracy": round(accuracy, 4),
        "AUC": round(auc, 4)
    }

    # Collect data for overall evaluation
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_proba.extend(y_proba)



# Convert metrics to DataFrame and sort
model_LifhtGBM = pd.DataFrame(metrics).T.sort_values(by="AUC", ascending=False)

# Calculate mean and median of all metrics across labels
overall_mean = model_LifhtGBM.astype(float).mean().round(4).to_dict()
overall_median = model_LifhtGBM.astype(float).median().round(4).to_dict()

# Add two summary rows: mean and median across all labels
model_LifhtGBM.loc["Overall (mean)"] = overall_mean
model_LifhtGBM.loc["Overall (median)"] = overall_median

# Display the final evaluation table
display(model_LifhtGBM)

,Positive Rate,TP Rate,FP Rate,Recall,Precision,F1 Score,Accuracy,AUC
NR-AhR,0.1174,0.7292,0.2708,0.4459,0.7292,0.5534,0.9155,0.8811
SR-MMP,0.1581,0.6721,0.3279,0.4385,0.6721,0.5307,0.8774,0.8540
NR-AR-LBD,0.0348,0.7222,0.2778,0.5417,0.7222,0.6190,0.9768,0.8325
SR-p53,0.0623,0.8125,0.1875,0.1512,0.8125,0.2549,0.9450,0.8325
SR-ATAD5,0.0367,0.8182,0.1818,0.1698,0.8182,0.2812,0.9682,0.8095
SR-ARE,0.1619,0.7031,0.2969,0.2344,0.7031,0.3516,0.8600,0.7907
NR-Aromatase,0.0514,0.8095,0.1905,0.2787,0.8095,0.4146,0.9596,0.7881
NR-ER-LBD,0.0500,0.6562,0.3438,0.2958,0.6562,0.4078,0.9571,0.7690
NR-PPAR-gamma,0.0289,0.6667,0.3333,0.1579,0.6667,0.2553,0.9734,0.7441
SR-HSE,0.0577,0.7500,0.2500,0.1974,0.7500,0.3125,0.9499,0.7316


## Deep Learning

### 1. GCN

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.metrics import roc_auc_score
from rdkit import Chem, RDLogger
from sklearn.model_selection import train_test_split
# import deepchem as dc
from rdkit.Chem import rdchem
from rdkit.Chem import AllChem
from mordred import Calculator, descriptors
from rdkit import Chem
from rdkit import DataStructs


RDLogger.DisableLog('rdApp.warning')  


In [ ]:
# Configuration parameters
BATCH_SIZE = 128
EPOCHS = 15
LEARNING_RATE = 0.001
HIDDEN_DIM = 256
DROPOUT = 0.7
TARGET_COLS = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 
              'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE',
              'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

In [ ]:
# 1. Data preprocessing (each task is processed separately)

def is_valid_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None and mol.GetNumAtoms() > 1
    except Exception:
        return False

def load_single_task_data(path,target_col):

    df = pd.read_csv(path)
    
    # Filter the missing values of the current task
    df = df.dropna(subset=[target_col])
    print(f"\nTask {target_col} Valid sample size: {len(df)}")
       
    # Filter the valid SMILES and retain the index
    valid_smiles_indices = df['smiles'].apply(is_valid_smiles)
    
    # Filter the samples whose target column values are 0 or 1
    valid_target_indices = df[target_col].isin([0, 1])
    
    # Merge the two conditions to obtain the final valid index
    valid_indices = valid_smiles_indices & valid_target_indices
    valid_df = df[valid_indices]
    
    # Extract the valid SMILES and target values
    valid_smiles = valid_df['smiles'].values
    y = valid_df[target_col].values.astype(np.float32)
    
    return valid_smiles, y

In [ ]:
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Chiral encoding mapping dictionary
    cip_code_mapping = {'R': 0, 'S': 1, 'r': 2, 's': 3, '': -1}
    
    # Atomic feature extraction
    atom_features = []
    for atom in mol.GetAtoms():
        # Handle chiral coding (numerical)
        cip_code = atom.GetProp("_CIPCode") if atom.HasProp("_CIPCode") else ""
        cip_value = cip_code_mapping.get(cip_code.upper(), -1)  # Convert to numerical value
        
        features = [
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetTotalDegree(),
            atom.GetImplicitValence(),
            atom.GetExplicitValence(),
            int(atom.GetIsAromatic()),
            atom.GetHybridization().real,
            atom.GetTotalNumHs(),
            int(atom.IsInRing()),
            atom.GetMass() / 100.0,
            atom.GetNumRadicalElectrons(),
            atom.GetChiralTag().real,
            cip_value,  # Chiral features encoded numerically
            atom.GetFormalCharge(),
            atom.GetNumExplicitHs(),
            atom.GetIsotope(),
            int(atom.HasOwningMol()),
            *[int(atom.IsInRingSize(i)) for i in range(3, 8)],
        ]
        atom_features.append(features)
        
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
        arr = np.zeros((1024,), dtype=int)
        DataStructs.ConvertToNumpyArray(fp, arr)
        
        atom_features[-1].extend(arr)  # Add the fingerprint feature to the atomic feature

    
    # Edge feature extraction
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])
        
        bond_features = [
            bond.GetBondTypeAsDouble(),
            int(bond.IsInRing()),
            int(bond.GetIsConjugated()),
            bond.GetBondDir().real,
            bond.GetStereo().real,
            bond.GetBondTypeAsDouble() * int(bond.IsInRing()),
            int(bond.GetIsAromatic()),
        ]
        edge_attr.extend([bond_features, bond_features])
    


    return Data(
        x=torch.tensor(atom_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float)
    )

In [ ]:
# 3. Custom dataset class
class SingleTaskDataset(Dataset):
    def __init__(self, smiles_list, y):
        super().__init__()
        self.smiles_list = smiles_list
        self.y = torch.tensor(y, dtype=torch.float)
        
    def len(self):
        return len(self.smiles_list)
    
    def get(self, idx):
        data = smiles_to_graph(self.smiles_list[idx])
        data.y = self.y[idx].unsqueeze(0)  # Maintain the two-dimensional shape
        return data

In [ ]:
class SingleTaskGCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, edge_dim):
        super().__init__()
        # The first layer: Fuse edge features to node features
        self.edge_processor = torch.nn.Linear(edge_dim, input_dim)
        
        # Maintain the original structure
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, 1)

        # initialization parameter
        self._init_weights()

    def _init_weights(self):
        for module in [self.edge_processor, self.conv1, self.conv2, self.lin]:
            if hasattr(module, 'weight'):
                torch.nn.init.xavier_uniform_(module.weight)
            if hasattr(module, 'bias') and module.bias is not None:
                module.bias.data.fill_(0.01)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        
        # === Edge feature fusion (without changing the input structure ===
        # Map the edge features to the node dimension
        edge_emb = self.edge_processor(edge_attr)
        
        # Enhance the node features by aggregating edge features
        row, col = edge_index
        aggregated_edge = torch.zeros_like(x).index_add_(0, col, edge_emb)
        x = x + aggregated_edge  # Keep the x-dimension unchanged

        # === Maintain the original GCN structure ===
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=DROPOUT, training=self.training)
        
        # The second layer uses edge weights
        edge_weight = edge_attr.mean(dim=1)  # Convert the multi-dimensional edge features into scalar weights
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        
        # === Maintain the original pooled structure ===
        if hasattr(data, 'batch') and data.batch is not None:
            x = global_mean_pool(x, data.batch)
        else:
            x = x.mean(dim=0, keepdim=True)
            
        x = self.lin(x)
        return x.squeeze(-1)

In [ ]:
# 5. Single-task training process
def train_single_task(target_col):
    # Load data
    smiles, y = load_single_task_data("tox21_cleaned.csv", target_col)
    
    # Split data into train, validation, and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        smiles, y, test_size=0.2, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(
        X_test, y_test, test_size=0.5, random_state=42)
    
    # Create datasets
    train_dataset = SingleTaskDataset(X_train.tolist(), y_train)
    val_dataset = SingleTaskDataset(X_val.tolist(), y_val)
    test_dataset = SingleTaskDataset(X_test.tolist(), y_test)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    
    # Initialize model, optimizer, and loss function
    model = SingleTaskGCN(
        input_dim=train_dataset[0].x.shape[1],
        hidden_dim=HIDDEN_DIM,
        edge_dim=7       # Edge feature dimension (New parameter)
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = torch.nn.BCEWithLogitsLoss()
    
    best_auc = 0
    for epoch in range(EPOCHS):
        # Train
        model.train()
        total_loss = 0
        for data in train_loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Validate
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for data in val_loader:
                pred = model(data)
                val_preds.append(pred.sigmoid().cpu().numpy())
                val_labels.append(data.y.cpu().numpy())
        
        val_auc = roc_auc_score(np.concatenate(val_labels), 
                               np.concatenate(val_preds))
        print(f"Task {target_col} | Epoch {epoch+1}/{EPOCHS} | "
              f"Train Loss: {total_loss/len(train_loader):.4f} | "
              f"Val AUC: {val_auc:.4f}")
        
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), f"best_model_{target_col}.pth")
    
    # Test
    model.load_state_dict(torch.load(f"best_model_{target_col}.pth"))
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for data in test_loader:
            pred = model(data)
            test_preds.append(pred.sigmoid().cpu().numpy())
            test_labels.append(data.y.cpu().numpy())
    
    test_auc = roc_auc_score(np.concatenate(test_labels), 
                            np.concatenate(test_preds))
    print(f"\nTask {target_col} Test result:")
    print(f"Test AUC: {test_auc:.4f}")
    print("="*50)
    
    return test_auc

In [ ]:
if __name__ == "__main__":
    results = {}
    for target_col in TARGET_COLS:
        metrics = train_single_task(target_col)
        results[target_col] = metrics


Task NR-AR Valid sample size: 7432
Task NR-AR | Epoch 1/30 | Train Loss: 0.1627 | Val AUC: 0.8130
Task NR-AR | Epoch 2/30 | Train Loss: 0.1100 | Val AUC: 0.8537
Task NR-AR | Epoch 3/30 | Train Loss: 0.0990 | Val AUC: 0.8478
Task NR-AR | Epoch 4/30 | Train Loss: 0.0926 | Val AUC: 0.8337
Task NR-AR | Epoch 5/30 | Train Loss: 0.0780 | Val AUC: 0.8613
Task NR-AR | Epoch 6/30 | Train Loss: 0.0775 | Val AUC: 0.8574
Task NR-AR | Epoch 7/30 | Train Loss: 0.0603 | Val AUC: 0.8259
Task NR-AR | Epoch 8/30 | Train Loss: 0.0569 | Val AUC: 0.8535
Task NR-AR | Epoch 9/30 | Train Loss: 0.0655 | Val AUC: 0.8551
Task NR-AR | Epoch 10/30 | Train Loss: 0.0436 | Val AUC: 0.8207
Early stopping triggered after 10 epochs.
TEST AUC: 0.7910157830837717

Task NR-AR-LBD Valid sample size: 6895
Task NR-AR-LBD | Epoch 1/30 | Train Loss: 0.1461 | Val AUC: 0.6526
Task NR-AR-LBD | Epoch 2/30 | Train Loss: 0.0935 | Val AUC: 0.7803
Task NR-AR-LBD | Epoch 3/30 | Train Loss: 0.0893 | Val AUC: 0.7706
Task NR-AR-LBD | Epoc

d:\Anaconda\envs\chem311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TEST AUC: 0.8076615831517793

Task SR-MMP Valid sample size: 5914
Task SR-MMP | Epoch 1/30 | Train Loss: 0.4454 | Val AUC: 0.7766
Task SR-MMP | Epoch 2/30 | Train Loss: 0.3456 | Val AUC: 0.8441
Task SR-MMP | Epoch 3/30 | Train Loss: 0.3177 | Val AUC: 0.8729
Task SR-MMP | Epoch 4/30 | Train Loss: 0.2549 | Val AUC: 0.8877
Task SR-MMP | Epoch 5/30 | Train Loss: 0.2279 | Val AUC: 0.8989
Task SR-MMP | Epoch 6/30 | Train Loss: 0.1933 | Val AUC: 0.8974
Task SR-MMP | Epoch 7/30 | Train Loss: 0.1756 | Val AUC: 0.8999
Task SR-MMP | Epoch 8/30 | Train Loss: 0.1343 | Val AUC: 0.8891
Task SR-MMP | Epoch 9/30 | Train Loss: 0.1132 | Val AUC: 0.8897
Task SR-MMP | Epoch 10/30 | Train Loss: 0.0790 | Val AUC: 0.8928
Task SR-MMP | Epoch 11/30 | Train Loss: 0.0700 | Val AUC: 0.8872
Task SR-MMP | Epoch 12/30 | Train Loss: 0.0541 | Val AUC: 0.8856
Early stopping triggered after 12 epochs.
TEST AUC: 0.9127877237851663

Task SR-p53 Valid sample size: 6902
Task SR-p53 | Epoch 1/30 | Train Loss: 0.2632 | Val AUC

In [ ]:
# Transform to pandas DataFrame
results_df = pd.DataFrame(list(results.items()), columns=['Target', 'AUC'])

# Calculate mean and median
mean_auc = results_df['AUC'].mean()
median_auc = results_df['AUC'].median()

print(f"Mean AUC: {mean_auc:.4f}")
print(f"Median AUC: {median_auc:.4f}")

display(results)

Mean AUC: 0.8234
Median AUC: 0.8249


,Recall,Precision,F1 Score,Accuracy,AUC
NR-AR,0.4000,1.0000,0.5714,0.9717,0.7910
NR-AR-LBD,0.4762,0.7692,0.5882,0.9797,0.8993
NR-AhR,0.4783,0.5789,0.5238,0.9100,0.8888
NR-Aromatase,0.1379,0.5000,0.2162,0.9510,0.8578
NR-ER,0.2099,0.7083,0.3238,0.8873,0.7336
NR-ER-LBD,0.3611,0.8667,0.5098,0.9647,0.8682
NR-PPAR-gamma,0.2500,0.3333,0.2857,0.9619,0.7054
SR-ARE,0.4175,0.5375,0.4699,0.8361,0.7856
SR-ATAD5,0.2069,0.5455,0.3000,0.9612,0.8421
SR-HSE,0.0000,0.0000,0.0000,0.9315,0.8077


### 1.2. Pytorch-Lightning Integration for DeepChem GCN Models

In [ ]:
import deepchem as dc
from deepchem.models import GCNModel
import pandas as pd
import numpy as np
from rdkit import Chem
from sklearn.model_selection import train_test_split
from deepchem.feat import MolGraphConvFeaturizer
import pytorch_lightning as pl
import torch


In [ ]:

def is_valid_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None and mol.GetNumAtoms() > 1
    except Exception:
        return False

def load_single_task_data(target_col):
    """加载单个任务的数据，并仅保留目标值为 0 和 1 的样本"""
    df = pd.read_csv('tox21_cleaned.csv')
    
    # 过滤当前任务的缺失值
    df = df.dropna(subset=[target_col])
    print(f"\nTask {target_col} Valid sample size: {len(df)}")
       
 # 过滤有效的 SMILES 并保留索引
    valid_smiles_indices = df['smiles'].apply(is_valid_smiles)
    
    # 过滤目标列值为 0 或 1 的样本
    valid_target_indices = df[target_col].isin([0, 1])
    
    # 合并两个条件，得到最终有效索引
    valid_indices = valid_smiles_indices & valid_target_indices
    valid_df = df[valid_indices]
    
    # 提取有效的 SMILES 和目标值
    valid_smiles = valid_df['smiles'].values
    y = valid_df[target_col].values.astype(np.float32)
    
    return valid_smiles, y

In [ ]:
# prepare LightningDataModule
class SmilesDataset(torch.utils.data.Dataset):
    def __init__(self, smiles, labels):
        assert len(smiles) == len(labels)
        featurizer = dc.feat.MolGraphConvFeaturizer()
        X = featurizer.featurize(smiles).flatten()
        self._samples = dc.data.NumpyDataset(X=X, y=labels)
        
    def __len__(self):
        return len(self._samples)
        
    def __getitem__(self, index):
        return (
            self._samples.X[index],
            self._samples.y[index],
            self._samples.w[index],
        )
    
    
class SmilesDatasetBatch:
    def __init__(self, batch):
        X = [np.array([b[0] for b in batch])]
        y = [np.array([b[1] for b in batch])]
        w = [np.array([b[2] for b in batch])]
        self.batch_list = [X, y, w]
        
        
def collate_smiles_dataset_wrapper(batch):
    return SmilesDatasetBatch(batch)

class SmilesDatasetModule(pl.LightningDataModule):
    def __init__(self, train_smiles, train_labels, batch_size):
        super().__init__()
        self._train_smiles = train_smiles
        self._train_labels = train_labels

        self._batch_size = batch_size
        
    def setup(self, stage):
        self.train_dataset = SmilesDataset(
            self._train_smiles,
            self._train_labels,
        )
        

        
    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self._batch_size,
            collate_fn=collate_smiles_dataset_wrapper,
            shuffle=True,  
            drop_last=True,
        )
        
class testSmilesDatasetModule(pl.LightningDataModule):
    def __init__(self, test_smiles, test_labels, batch_size):
        super().__init__()
        self._test_smiles = test_smiles
        self._test_labels = test_labels
        self._batch_size = batch_size

    def setup(self, stage=None):
        self.test_dataset = SmilesDataset(
            self._test_smiles,
            self._test_labels,
        )

    def test_dataloader(self):  # add test_dataloader method
        return torch.utils.data.DataLoader(
            self.test_dataset,
            batch_size=self._batch_size,
            collate_fn=collate_smiles_dataset_wrapper,
            shuffle=False,  
            drop_last=False,  
        )

    def get_test_smiles_labels(self):  
        return self._test_smiles, self._test_labels

In [ ]:

class GCNModule(pl.LightningModule):
    def __init__(self, mode, n_tasks, learning_rate, hidden_dim, dropout):
        super().__init__()
        self.save_hyperparameters(
            "mode",
            "n_tasks",
            "learning_rate",
            "hidden_dim",
            "dropout",
        )
        self.gcn_model = GCNModel(
            mode=self.hparams.mode,
            n_tasks=self.hparams.n_tasks,
            learning_rate=self.hparams.learning_rate,
            hidden_dim=self.hparams.hidden_dim,
            dropout=self.hparams.dropout,
        )
        self.pt_model = self.gcn_model.model
        self.loss = self.gcn_model._loss_fn
        
    def configure_optimizers(self):
        return self.gcn_model.optimizer._create_pytorch_optimizer(
            self.pt_model.parameters(),
        )
    
    def training_step(self, batch, batch_idx):
        batch = batch.batch_list
        inputs, labels, weights = self.gcn_model._prepare_batch(batch)
        outputs = self.pt_model(inputs)
        
        if isinstance(outputs, torch.Tensor):
            outputs = [outputs]
    
        if self.gcn_model._loss_outputs is not None:
            outputs = [outputs[i] for i in self.gcn_model._loss_outputs]
    

            
        loss_outputs = self.loss(outputs, labels, weights)
        
        self.log(
            "train_loss",
            loss_outputs,
            on_epoch=True,
            sync_dist=True,
            reduce_fx="mean",
            prog_bar=True,
            batch_size=32
        )
        
        return loss_outputs
    
        # define test_step 
    def test_step(self, batch, batch_idx):
        smiles,labels = self.trainer.datamodule.get_test_smiles_labels()
    
        featurizer = MolGraphConvFeaturizer()
        X =featurizer.featurize(smiles)

        test_dataset = dc.data.NumpyDataset(X=X, y=labels)

        metrics = [
        dc.metrics.Metric(dc.metrics.roc_auc_score, name="AUC"),
        dc.metrics.Metric(dc.metrics.f1_score, name="F1"),
        dc.metrics.Metric(dc.metrics.recall_score, name="Recall"),
        dc.metrics.Metric(dc.metrics.accuracy_score, name="Accuracy")
        ]

        results = self.gcn_model.evaluate(test_dataset, metrics)
        performance_df = pd.DataFrame([results], columns=results.keys())
        Summary.append(performance_df.round(4))
        
        return results



In [ ]:
def train_single_task(target_col):
    smiles, y = load_single_task_data(target_col)
    X_train, X_test, y_train, y_test = train_test_split(
        smiles, y, test_size=0.3, random_state=2
    )

    # train dataset module
    smiles_datasetmodule = SmilesDatasetModule(
        train_smiles=X_train,
        train_labels=y_train,

        batch_size=32,
    )
    gcnmodule = GCNModule(
        n_tasks=1, 
        mode='classification',  
        hidden_dim=512,
        learning_rate=0.001,
        dropout=0.2,
    )

    trainer = pl.Trainer(
        max_epochs=20,
        accelerator='cpu',
    )

    # Traine
    trainer.fit(
        model=gcnmodule,
        datamodule=smiles_datasetmodule,
    )

    # test dataset model
    smiles_datasetmodule = testSmilesDatasetModule(

        test_smiles=X_test, 
        test_labels=y_test,
        batch_size=len(y_test),  
    )
    

    test_result = trainer.test(model=gcnmodule, datamodule=smiles_datasetmodule)

    return test_result



In [ ]:

TARGET_COLS = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 
              'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE',
              'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

if __name__ == "__main__":
    Summary = []
    for target in TARGET_COLS:
        train_single_task(target) 



Task NR-AR Valid sample size: 7432


You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type | Params | Mode 
------------------------------------------
0 | pt_model | GCN  | 29.4 K | train
------------------------------------------
29.4 K    Trainable params
0         Non-trainable params
29.4 K    Total params
0.118     Total estimated model params size (MB)
24        Modules in train mode
0         Modules in eval mode
d:\Anaconda\envs\chem311\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 19: 100%|██████████| 162/162 [00:12<00:00, 13.24it/s, v_num=130, train_loss_step=0.0898, train_loss_epoch=0.117]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 162/162 [00:12<00:00, 13.18it/s, v_num=130, train_loss_step=0.0898, train_loss_epoch=0.117]


...

Epoch 19: 100%|██████████| 150/150 [00:10<00:00, 14.26it/s, v_num=141, train_loss_step=0.182, train_loss_epoch=0.166] 

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 150/150 [00:10<00:00, 14.21it/s, v_num=141, train_loss_step=0.182, train_loss_epoch=0.166]


d:\Anaconda\envs\chem311\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 1/1 [00:11<00:00,  0.09it/s]


In [ ]:
#Merge all results into a single DataFrame
final_report = pd.concat(Summary, ignore_index=True)

final_report.insert(0, 'Target', TARGET_COLS[:len(Summary)])

mean_values = final_report.iloc[:, 1:].mean()

final_report.loc[len(final_report)] = ['Overall'] + mean_values.tolist()

# Output the final report
display(final_report.round(4))

,Target,AUC,F1,Recall,Accuracy
0,NR-AR,0.7605,0.5667,0.4048,0.9766
1,NR-AR-LBD,0.8298,0.5882,0.4667,0.9763
2,NR-AhR,0.8940,0.0637,0.0329,0.8825
3,NR-Aromatase,0.8459,0.0000,0.0000,0.9521
4,NR-ER,0.7173,0.0635,0.0328,0.8750
5,NR-ER-LBD,0.8105,0.1846,0.1034,0.9501
6,NR-PPAR-gamma,0.8556,0.0345,0.0175,0.9715
7,SR-ARE,0.7740,0.2089,0.1286,0.8292
8,SR-ATAD5,0.8341,0.0241,0.0122,0.9626
9,SR-HSE,0.7851,0.0661,0.0364,0.9426


### 2. D-MPNN

In [35]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [36]:
import pandas as pd
import chemprop
from chemprop import data, featurizers
import torch
import torch.nn as nn
from model.dmpnn_focal import MPNNModel_FocalLoss
from model.utils import compute_classification_report

In [3]:
tox21_tasks = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
               'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


tox21_data = pd.read_csv('../data/tox21.csv')
smiles = tox21_data.loc[:, 'smiles'].values
targets = tox21_data.loc[:, tox21_tasks].values
num_workers = 0

# Convert to Chemprop's MoleculeDatapoint format
all_data = [data.MoleculeDatapoint.from_smi(smile, target) for smile, target in zip(smiles, targets)]

# Transform into RDkit Mol objects for structure based splits
mols = [data.mol for data in all_data]
train_indices, val_indices, test_indices = data.make_split_indices(mols, "random", (0.7, 0.2, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(
    all_data, train_indices, val_indices, test_indices
)


# Featurize the data
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_data = data.MoleculeDataset(train_data[0], featurizer)
val_data = data.MoleculeDataset(val_data[0], featurizer)
test_data = data.MoleculeDataset(test_data[0], featurizer)

# Create dataloaders
train_loader = data.build_dataloader(train_data, num_workers=num_workers)
val_loader = data.build_dataloader(val_data, num_workers=num_workers, shuffle=False)
test_loader = data.build_dataloader(test_data, num_workers=num_workers, shuffle=False)

[20:11:25] WARNING: not removing hydrogen atom without neighbors
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
Dropping last batch of size 1 to avoid issues with batch normalization (dataset size = 1601, batch_size = 64)


### 2.1 Initial

In [33]:
from lightning import pytorch as pl

mp = chemprop.nn.BondMessagePassing()
agg = chemprop.nn.MeanAggregation()
ffn = chemprop.nn.BinaryClassificationFFN(n_tasks=len(tox21_tasks))

batch_norm = False
metric_list = None   # AUROC used by default
mpnn = chemprop.models.MPNN(mp, agg, ffn, batch_norm, metric_list)

trainer_mpnn = pl.Trainer(
    logger=False,
    enable_checkpointing=False, # Use `True` if you want to save model checkpoints. The checkpoints will be saved in the `checkpoints` folder.
    enable_progress_bar=True,
    accelerator="cpu",
    devices=1,
    max_epochs=20, # number of epochs to train for
)

trainer_mpnn.fit(mpnn, train_loader)


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
Loading `train_dataloader` to estimate number of stepping batches.
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

  | Name            | Type                    | Params | Mode 
----------------------------------------------

Epoch 19: 100%|██████████| 88/88 [00:10<00:00,  8.50it/s, train_loss_step=0.179, train_loss_epoch=0.177]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 88/88 [00:10<00:00,  8.49it/s, train_loss_step=0.179, train_loss_epoch=0.177]


#### · Performancce on Validation Set

In [34]:
trainer_mpnn = pl.Trainer(logger=True)  # default TensorBoard logger
trainer_mpnn.validate(model=mpnn, dataloaders=val_loader)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 25/25 [00:04<00:00,  5.23it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/roc          │    0.8718467950820923     │
│         val_loss          │    0.1882823258638382     │
└───────────────────────────┴───────────────────────────┘

[{'val/roc': 0.8718467950820923, 'val_loss': 0.1882823258638382}]

In [37]:
import torch

pred_probs = trainer_mpnn.predict(mpnn, dataloaders=val_loader)[:-1]
probs = torch.cat(pred_probs, dim=0).cpu().numpy()

all_y = []
for batch in val_loader:
    y = batch.Y  # 👈 get the label tensor from TrainingBatch
    all_y.append(y)
y_true = torch.cat(all_y, dim=0).cpu().numpy()  # shape: (num_samples, num_tasks)

model0_report_val = compute_classification_report(y_true, probs, threshold=0.5, task_cols=tox21_tasks)
model0_report_val


/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 26/26 [00:02<00:00, 10.83it/s]
 - Precision: mean = 0.6030, median = 0.7115
 - Recall:    mean = 0.2081, median = 0.2033
 - F1:        mean = 0.2944, median = 0.3211
 - AUC:       mean = 0.8427, median = 0.8372


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,NR-AR,0.931034,0.450000,0.606742,0.814533,60,29,1501
1,NR-AR-LBD,0.888889,0.400000,0.551724,0.870976,60,27,1400
2,NR-AhR,0.728261,0.413580,0.527559,0.909410,162,92,1349
3,NR-Aromatase,0.000000,0.000000,0.000000,0.868866,56,0,1200
4,NR-ER,0.701754,0.235294,0.352423,0.737772,170,57,1269
5,NR-ER-LBD,0.875000,0.175000,0.291667,0.823091,80,16,1429
6,NR-PPAR-gamma,0.000000,0.000000,0.000000,0.834313,40,0,1342
7,SR-ARE,0.721311,0.231579,0.350598,0.833951,190,61,1184
8,SR-ATAD5,0.500000,0.018519,0.035714,0.840134,54,2,1470
9,SR-HSE,0.636364,0.088608,0.155556,0.822970,79,11,1309


#### · Performancce on Test Set

In [38]:
trainer_mpnn.test(mpnn, test_loader)

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 13/13 [00:02<00:00,  5.17it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/roc          │     0.864683210849762     │
└───────────────────────────┴───────────────────────────┘

[{'test/roc': 0.864683210849762}]

In [39]:
pred_probs = trainer_mpnn.predict(mpnn, dataloaders=test_loader)
probs = torch.cat(pred_probs, dim=0).cpu().numpy()
all_y = []
for batch in test_loader:
    y = batch.Y  # 👈 get the label tensor from TrainingBatch
    all_y.append(y)
y_true = torch.cat(all_y, dim=0).cpu().numpy()  # shape: (num_samples, num_tasks)

model0_report_test = compute_classification_report(y_true, probs, threshold=0.5, task_cols=tox21_tasks)
model0_report_test


/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 13/13 [00:01<00:00,  6.91it/s]
 - Precision: mean = 0.5062, median = 0.6237
 - Recall:    mean = 0.1926, median = 0.1657
 - F1:        mean = 0.2675, median = 0.2650
 - AUC:       mean = 0.8327, median = 0.8497


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,NR-AR,0.846154,0.343750,0.488889,0.696002,32,13,734
1,NR-AR-LBD,0.636364,0.388889,0.482759,0.852744,18,11,682
2,NR-AhR,0.690476,0.376623,0.487395,0.907880,77,42,667
3,NR-Aromatase,0.000000,0.000000,0.000000,0.846632,36,0,577
4,NR-ER,0.611111,0.135802,0.222222,0.762612,81,18,615
5,NR-ER-LBD,0.777778,0.205882,0.325581,0.860261,34,9,701
6,NR-PPAR-gamma,0.000000,0.000000,0.000000,0.860711,17,0,634
7,SR-ARE,0.720000,0.195652,0.307692,0.800081,92,25,599
8,SR-ATAD5,0.000000,0.000000,0.000000,0.812503,31,0,708
9,SR-HSE,0.500000,0.085714,0.146341,0.859584,35,6,667


### 2.2 With Focal Loss

In [40]:
import pytorch_lightning as pl

In [41]:
alphas = []
for task in tox21_tasks:
    pos = tox21_data[task].sum()
    total = tox21_data[task].notna().sum()
    alpha = 1 - (pos / total)
    alphas.append(alpha)

# Create a tensor
alpha_tensor = torch.tensor(alphas, dtype=torch.float32)

In [43]:
mp = chemprop.nn.BondMessagePassing()
agg = chemprop.nn.MeanAggregation()
ffn = nn.Sequential(
    nn.Linear(300, 300),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(300, 12)
    )

# ffn = chemprop.nn.BinaryClassificationFFN(n_tasks=len(tox21_tasks), dropout=0.7)
        
batch_norm = False
metric_list = None
mpnn_focal = MPNNModel_FocalLoss(mp, agg, ffn, batch_norm, metric_list, alpha_tensor, gamma=2.5)
mpnn_focal.load_state_dict(torch.load("../model/mpnn_focal_model_new.pt"))


<All keys matched successfully>

#### · Performancce on Validation Set

In [44]:
trainer_mpnn_focal = pl.Trainer()
trainer_mpnn_focal.validate(mpnn_focal, val_loader)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 25/25 [00:01<00:00, 18.88it/s]Best thresholds: [np.float64(0.6473684210526316), np.float64(0.6473684210526316), np.float64(0.5210526315789473), np.float64(0.5631578947368421), np.float64(0.5210526315789473), np.float64(0.6052631578947368), np.float64(0.5210526315789473), np.float64(0.5210526315789473), np.float64(0.6473684210526316), np.float64(0.5631578947368421), np.float64(0.5210526315789473), np.float64(0.5210526315789473)]
Best thresholds: [np.float64(0.6473684210526316), np.float64(0.6473684210526316), np.float64(0.5210526315789473), np.float64(0.5631578947368421), np.float64(0.5210526315789473), np.float64(0.6052631578947368), np.float64(0.5210526315789473), np.float64(0.5210526315789473), np.float64(0.6473684210526316), np.float64(0.5631578947368421), np.float64(0.5210526315789473), np.float64(0.5210526315789473)]
Validation DataLoader 0: 100%|██████████| 25/25 [00:01<00:00, 15.29it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val_mean_auc        │    0.8402740955352783     │
└───────────────────────────┴───────────────────────────┘

[{'val_mean_auc': 0.8402740955352783}]

In [45]:
preds = trainer_mpnn_focal.predict(mpnn_focal, dataloaders=val_loader)[:-1]

probs = torch.cat([r["probs"] for r in preds], dim=0).cpu().numpy()
y_true = torch.cat([r["targets"] for r in preds], dim=0).cpu().numpy()

model1_report_val = compute_classification_report(y_true, probs, threshold=0.55, task_cols=tox21_tasks)
model1_report_val

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 26/26 [00:01<00:00, 20.83it/s]
 - Precision: mean = 0.4106, median = 0.4256
 - Recall:    mean = 0.4852, median = 0.5242
 - F1:        mean = 0.4308, median = 0.4347
 - AUC:       mean = 0.8403, median = 0.8452


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,NR-AR,0.534483,0.516667,0.525424,0.807321,60,58,1501
1,NR-AR-LBD,0.551724,0.533333,0.542373,0.859876,60,58,1400
2,NR-AhR,0.567568,0.648148,0.605187,0.895332,162,185,1349
3,NR-Aromatase,0.272727,0.321429,0.295082,0.841174,56,66,1200
4,NR-ER,0.500000,0.382353,0.433333,0.734481,170,130,1269
5,NR-ER-LBD,0.351145,0.575000,0.436019,0.808970,80,131,1429
6,NR-PPAR-gamma,0.269841,0.425000,0.330097,0.874770,40,63,1342
7,SR-ARE,0.506250,0.426316,0.462857,0.842608,190,160,1184
8,SR-ATAD5,0.198758,0.592593,0.297674,0.868618,54,161,1470
9,SR-HSE,0.264151,0.531646,0.352941,0.824812,79,159,1309


#### · Performance on Test set

In [46]:
trainer_mpnn_focal.test(mpnn_focal, test_loader)

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 14.81it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_mean_auc       │     0.830152690410614     │
└───────────────────────────┴───────────────────────────┘

[{'test_mean_auc': 0.830152690410614}]

In [47]:
preds = trainer_mpnn_focal.predict(mpnn_focal, dataloaders=test_loader)

probs = torch.cat([r["probs"] for r in preds], dim=0).cpu().numpy()
y_true = torch.cat([r["targets"] for r in preds], dim=0).cpu().numpy()

model1_report_test = compute_classification_report(y_true, probs, threshold=0.55, task_cols=tox21_tasks)
model1_report_test

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 25.63it/s]
 - Precision: mean = 0.4018, median = 0.4019
 - Recall:    mean = 0.4562, median = 0.4202
 - F1:        mean = 0.4095, median = 0.3970
 - AUC:       mean = 0.8302, median = 0.8464


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,NR-AR,0.478261,0.343750,0.400000,0.688479,32,23,734
1,NR-AR-LBD,0.370370,0.555556,0.444444,0.853999,18,27,682
2,NR-AhR,0.518519,0.545455,0.531646,0.914792,77,81,667
3,NR-Aromatase,0.433333,0.361111,0.393939,0.836568,36,30,577
4,NR-ER,0.534884,0.283951,0.370968,0.735493,81,43,615
5,NR-ER-LBD,0.339286,0.558824,0.422222,0.848664,34,56,701
6,NR-PPAR-gamma,0.227273,0.294118,0.256410,0.860807,17,22,634
7,SR-ARE,0.433735,0.391304,0.411429,0.809236,92,83,599
8,SR-ATAD5,0.191176,0.419355,0.262626,0.816887,31,68,708
9,SR-HSE,0.277108,0.657143,0.389831,0.844078,35,83,667


### 3. RNN - BiLSTM

In [3]:
import os, random, numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             precision_score, recall_score)
from rdkit import Chem
from tqdm.auto import tqdm     

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1. Load the data

In [4]:

# ========== 0. Data path & Task column name ============================================
CSV_PATH = "../data/tox21.csv"                       
targets = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER',
           'NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5',
           'SR-HSE','SR-MMP','SR-p53']

# ========== 1. Load the data ==============================================
df = pd.read_csv(CSV_PATH)
base_smiles = df["smiles"].astype(str).tolist()
y_base      = df[targets].fillna(-1).astype('float32').values  # -1 sentinel



##### 2. Data Manipulation and Model Preparation

In [ ]:
# ========== 2. Random-SMILES data augmentation ====================================
AUG_FACTOR = 3          # The number of additional random SMILES for each molecule can be adjusted from 0 to 5
aug_smiles, aug_labels = [], []

if AUG_FACTOR > 0:
    print(f" Augmenting with Random-SMILES ×{AUG_FACTOR} …")
for s, y in tqdm(list(zip(base_smiles, y_base)), total=len(base_smiles)):
    aug_smiles.append(s); aug_labels.append(y)
    if AUG_FACTOR == 0: continue
    mol = Chem.MolFromSmiles(s)
    if mol is None:           # Skip the augmentation if it cannot be parsed
        continue
    for _ in range(AUG_FACTOR):
        rs = Chem.MolToSmiles(mol, doRandom=True)
        aug_smiles.append(rs); aug_labels.append(y)

smiles = aug_smiles
y_all  = np.vstack(aug_labels)

print(f"Dataset size after augmentation: {len(smiles)} SMILES")


⏫ Augmenting with Random-SMILES ×3 …


100%|██████████| 8006/8006 [00:05<00:00, 1473.08it/s]

Dataset size after augmentation: 32024 SMILES


In [6]:
# ========== 3. Character level Tokenizer & Padding ================================
tok = tf.keras.preprocessing.text.Tokenizer(char_level=True)
tok.fit_on_texts(smiles)
seqs = tok.texts_to_sequences(smiles)

MAXLEN = 150
X = tf.keras.preprocessing.sequence.pad_sequences(
        seqs, maxlen=MAXLEN, padding="post", truncating="post")

VOCAB = len(tok.word_index) + 1


##### 3. Build the Model and Fine Tunning

In [8]:
# ========== 4. Split the data with seed =====================
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y_all, test_size=0.30, random_state=SEED, shuffle=True)
X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=SEED, shuffle=True)

# ========== 5. Two-Layer BiLSTM + Masked Self-Attention ==================
class MaskedAttention(tf.keras.layers.Layer):
    def __init__(self): super().__init__()
    def build(self, input_shape):
        self.score = tf.keras.layers.Dense(1, activation='tanh')
    def call(self, x, mask=None):          # x:[B,T,H]
        e = tf.squeeze(self.score(x), axis=-1)       # [B,T]
        if mask is not None:
            e -= 1e9 * (1 - tf.cast(mask, tf.float32))   # mask pad
        w = tf.nn.softmax(e, axis=1)                    # attention weights
        out = tf.reduce_sum(x * tf.expand_dims(w, -1), axis=1) # [B,H]
        return out

def build_model(vocab, emb_dim=128, hid=64, dropout=0.3):
    inp = tf.keras.Input(shape=(MAXLEN,), dtype='int32')
    emb = tf.keras.layers.Embedding(vocab, emb_dim, mask_zero=True)(inp)
    x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(hid, return_sequences=True,
                                 dropout=dropout))(emb)
    x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(hid, return_sequences=True,
                                 dropout=dropout))(x)
    att = MaskedAttention()(x)
    out = tf.keras.layers.Dense(len(targets), activation='sigmoid')(att)
    return tf.keras.Model(inp, out)

model = build_model(VOCAB)

# ========== 6. Masked Focal Loss (γ=2, α=0.25) ===========================
def masked_focal(y_true, y_pred, γ=2.0, α=0.25):
    valid = tf.not_equal(y_true, -1.0)
    y_clean = tf.where(valid, y_true, 0.)
    pt = y_clean * y_pred + (1 - y_clean) * (1 - y_pred)
    loss = -α * tf.pow(1 - pt, γ) * tf.math.log(tf.clip_by_value(pt, 1e-7, 1.0))
    loss = tf.where(valid, loss, 0.)
    return tf.reduce_sum(loss) / tf.reduce_sum(tf.cast(valid, tf.float32))

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=5.0),
              loss=masked_focal)

early = tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)

# ========== 7. Fit the model =======================================================
model.fit(X_tr, y_tr, epochs=15, batch_size=128,
          validation_data=(X_va, y_va), callbacks=[early], verbose=2)

# ========== 8. Search for the optimal threshold of each task based on the validation set ===============================
val_prob = model.predict(X_va, batch_size=256, verbose=0)
mask_va = (y_va == -1)
best_thr = []
for i in range(len(targets)):
    v = ~mask_va[:, i]
    if v.sum() == 0:
        best_thr.append(0.5); continue
    yt, yp = y_va[v, i], val_prob[v, i]
    ts = np.linspace(0.1, 0.9, 17)
    f1s = [f1_score(yt, yp >= t, zero_division=0) for t in ts]
    best_thr.append(ts[int(np.argmax(f1s))])


Epoch 1/15


c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\layer.py:932: UserWarning: Layer 'masked_attention_1' (of type MaskedAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


176/176 - 266s - 2s/step - loss: 0.0174 - val_loss: 0.0157
Epoch 2/15
176/176 - 251s - 1s/step - loss: 0.0157 - val_loss: 0.0153
Epoch 3/15
176/176 - 255s - 1s/step - loss: 0.0154 - val_loss: 0.0151
Epoch 4/15
176/176 - 284s - 2s/step - loss: 0.0151 - val_loss: 0.0149
Epoch 5/15
176/176 - 308s - 2s/step - loss: 0.0149 - val_loss: 0.0148
Epoch 6/15
176/176 - 402s - 2s/step - loss: 0.0147 - val_loss: 0.0146
Epoch 7/15
176/176 - 338s - 2s/step - loss: 0.0144 - val_loss: 0.0145
Epoch 8/15
176/176 - 335s - 2s/step - loss: 0.0142 - val_loss: 0.0142
Epoch 9/15
176/176 - 309s - 2s/step - loss: 0.0140 - val_loss: 0.0139
Epoch 10/15
176/176 - 410s - 2s/step - loss: 0.0137 - val_loss: 0.0139
Epoch 11/15
176/176 - 294s - 2s/step - loss: 0.0135 - val_loss: 0.0137
Epoch 12/15
176/176 - 285s - 2s/step - loss: 0.0133 - val_loss: 0.0136
Epoch 13/15
176/176 - 299s - 2s/step - loss: 0.0131 - val_loss: 0.0136
Epoch 14/15
176/176 - 293s - 2s/step - loss: 0.0129 - val_loss: 0.0133
Epoch 15/15
176/176 - 298s

##### 4. Model Evaluation

In [9]:
# ========== 9. Test Set Evaluation ======================================
test_prob = model.predict(X_te, batch_size=256, verbose=0)
mask_te   = (y_te == -1)

rows = []
for i, col in enumerate(targets):
    v = ~mask_te[:, i]
    if v.sum() == 0: continue
    yt, yp = y_te[v, i], test_prob[v, i]
    y_bin = (yp >= best_thr[i]).astype(int)
    auc = roc_auc_score(yt, yp) if len(np.unique(yt))==2 else np.nan
    acc = accuracy_score(yt, y_bin)
    f1  = f1_score(yt, y_bin, zero_division=0)
    pre = precision_score(yt, y_bin, zero_division=0)
    rec = recall_score(yt, y_bin, zero_division=0)
    rows.append([col, auc, acc, f1, pre, rec])

df = pd.DataFrame(rows, columns=["Target","AUC","Accuracy","F1","Precision","Recall"])
# overall mean / median
stats = ["AUC","Accuracy","F1","Precision","Recall"]
df_mean   = df[stats].mean().to_frame().T
df_median = df[stats].median().to_frame().T
df_mean.insert(0,"Target","Overall Mean")
df_median.insert(0,"Target","Overall Median")
model_BiLSTM = pd.concat([df, df_mean, df_median], ignore_index=True)

print(f"\n=== Test-set metrics ===")
print(model_BiLSTM)


=== Test-set metrics ===
            Target       AUC  Accuracy        F1  Precision    Recall
0            NR-AR  0.821987  0.973100  0.634146   0.759124  0.544503
1        NR-AR-LBD  0.905035  0.973945  0.608696   0.705882  0.535032
2           NR-AhR  0.873219  0.848386  0.504092   0.413978  0.644351
3     NR-Aromatase  0.851497  0.883343  0.339257   0.258621  0.492958
4            NR-ER  0.749675  0.878950  0.420513   0.520635  0.352688
5        NR-ER-LBD  0.873452  0.938627  0.444915   0.415020  0.479452
6    NR-PPAR-gamma  0.854608  0.965781  0.255556   0.338235  0.205357
7           SR-ARE  0.803797  0.755618  0.508475   0.396825  0.707547
8         SR-ATAD5  0.845788  0.917566  0.294695   0.215517  0.465839
9           SR-HSE  0.834169  0.927916  0.355353   0.386139  0.329114
10          SR-MMP  0.869898  0.848998  0.563977   0.541471  0.588435
11          SR-p53  0.830056  0.824757  0.309751   0.205323  0.630350
12    Overall Mean  0.842765  0.894749  0.436619   0.429731  0.4